# Data prep — step by step

All logic lives in `scripts/build_datasets.py`. This notebook walks through each step.

Or run everything at once: `python scripts/build_datasets.py`

## Step 1 — Load Superstore (US transactions)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.build_datasets import load_superstore

superstore = load_superstore()
print(superstore.shape)
print("Regions:", superstore["region"].unique())
superstore.head(3)

## Step 2 — Add US context (holidays, CPI, weather)

In [ ]:
from scripts.build_datasets import add_context, load_macro, load_weather

with_context = add_context(superstore, load_macro(), load_weather())
print("Added columns:", [c for c in with_context.columns if c not in superstore.columns])
with_context[["order_date", "region", "is_us_federal_holiday", "cpi_index", "temp_c"]].head(3)

## Step 3 — Add synthetic loyalty CRM

In [ ]:
from scripts.build_datasets import add_crm

with_context["transaction_id"] = [f"txn_{i:07d}" for i in range(len(with_context))]
fact = add_crm(with_context)
fact[["transaction_id", "customer_id", "sales", "tier", "points_balance"]].head(3)

## Step 4 — Build customer table (group by customer_id)

In [ ]:
from scripts.build_datasets import build_dim_customers

dim = build_dim_customers(fact)
print(dim.shape)
dim.head(3)

## Step 5 — Telco + Instacart (separate customer IDs, not joined to Superstore)

In [ ]:
from scripts.build_datasets import build_telco_customers, build_instacart_users

telco = build_telco_customers()
print("Telco:", telco.shape, "| churn rate:", telco["churn_flag"].mean().round(3))

instacart = build_instacart_users()
if instacart is not None:
    print("Instacart users:", instacart.shape)
else:
    print("Instacart skipped (no Kaggle credentials)")

## Step 6 — Save final datasets

In [ ]:
from scripts.build_datasets import run

outputs = run()
for name, path in outputs.items():
    df = pd.read_csv(path)
    print(f"{name:20s} {len(df):>8,} rows  →  {path}")